In [3]:
from dotenv import load_dotenv
import os

load_dotenv()

api = os.getenv("PINECONE_DB_API")


In [4]:
from langchain_community.retrievers import PineconeHybridSearchRetriever


In [5]:
from pinecone import Pinecone, ServerlessSpec

index_name = "hybrid-search"
pc = Pinecone(api_key=api)

c:\anaconda3\envs\gen_env\lib\site-packages\pinecone\data\index.py:1: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [6]:


# creating index
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="dotproduct",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

In [7]:
curr_index = pc.Index(index_name)
curr_index

In [9]:
from langchain_huggingface import HuggingFaceEmbeddings

In [10]:
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

In [12]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
embeddings

HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [15]:
from pinecone_text.sparse import BM25Encoder
bm25_encoder = BM25Encoder().default()
bm25_encoder

In [18]:
# import nltk
# nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to C:\Users\Kushaagra
[nltk_data]     Mehta\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

In [21]:
sentences = [
    "In 2023, i visited the Eiffel Tower.",
    "In 2022 I visited Paris.",
    "In 2021 I visited France.",
]

bm25_encoder.fit(sentences)
bm25_encoder.dump("values.json")
bm25_encoder= BM25Encoder().load("values.json")

100%|██████████| 3/3 [00:00<00:00, 3000.93it/s]


In [23]:
retriever = PineconeHybridSearchRetriever(embeddings=embeddings, sparse_encoder=bm25_encoder, index=curr_index)
retriever

PineconeHybridSearchRetriever(embeddings=HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False), sparse_encoder=<pinecone_text.sparse.bm25_encoder.BM25Encoder object at 0x00000221A74CB100>, index=<pinecone.data.index.Index object at 0x00000221D091B8E0>)

In [24]:
retriever.add_texts(sentences)

100%|██████████| 1/1 [00:02<00:00,  2.53s/it]


In [25]:
retriever.invoke("Which city did I visit last")

[Document(metadata={'score': 0.232257754}, page_content='In 2022 I visited Paris.'),
 Document(metadata={'score': 0.221372217}, page_content='In 2021 I visited France.'),
 Document(metadata={'score': 0.194383472}, page_content='In 2023, i visited the Eiffel Tower.')]

In [26]:
retriever.invoke("Which city did I visit first")


[Document(metadata={'score': 0.224311531}, page_content='In 2022 I visited Paris.'),
 Document(metadata={'score': 0.212401509}, page_content='In 2021 I visited France.'),
 Document(metadata={'score': 0.189828262}, page_content='In 2023, i visited the Eiffel Tower.')]